In [1]:
import sys
from scipy.io import mmread
import os
import glob
import pandas as pd
import numpy as np
#from pandas_ods_reader import read_ods
from copy import deepcopy
import pprint
import json
import re
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import HuberRegressor
from sklearn import preprocessing
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.distance import pdist
from scipy.spatial.distance import squareform
from sklearn.manifold import TSNE
from sklearn import metrics
from sklearn.cluster import DBSCAN
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from collections import Counter
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage
import harmonypy as hm
from matplotlib.cm import ScalarMappable
from datetime import date
import mpld3
import hvplot.pandas
import holoviews as hv
from holoviews import opts
import panel as pn
import bokeh
from bokeh.resources import INLINE
from adjustText import adjust_text
from scipy.stats import mannwhitneyu, false_discovery_control, wilcoxon
import pygwalker as pyg
import matplotlib as mpl 

import dimorph_processing as dp
import cell_comparison as cc
import sex_stats as ss

today = str(date.today())
%matplotlib notebook
%load_ext autoreload
%autoreload 2

#change matplotlib font type to make compatibile with illustrator
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

In [2]:
#cluster_fn = 'Vglut2-6-Otp-Zic5'
run = '311224_run'
cell_class = 'gaba'
delta_data_folder = '/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run+'/gene_delta_plots/data/'
#test_df = pd.read_json(delta_data_folder + cell_class + '_expr_mlog_df_c_' +cluster_fn +'.json')

In [3]:
# Change the current working directory to the specified path
os.chdir(delta_data_folder)

# Get a list of all files in the directory that include '_expr_mlog_df_c_'
files = [f for f in os.listdir() if '_expr_mlog_df_c_' in f]

# Print the list of files
print(files)

['gaba_expr_mlog_df_c_GABA-13-Fign-Lrpprc.json', 'gaba_expr_mlog_df_c_GABA-16-Lmo1-Chn2.json', 'gaba_expr_mlog_df_c_GABA-29-Lpyd1-Unc13c.json', 'gaba_expr_mlog_df_c_GABA-33-Cartpt-Unc13c.json', 'gaba_expr_mlog_df_c_GABA-24-Prlr-St18.json', 'gaba_expr_mlog_df_c_GABA-20-Meis2-Foxp2.json', 'gaba_expr_mlog_df_c_GABA-22-Col23a1-Hs3st4.json', 'gaba_expr_mlog_df_c_GABA-2-Sst-Npy-Maf.json', 'gaba_expr_mlog_df_c_GABA-17-Isl1-Pou3f2.json', 'gaba_expr_mlog_df_c_GABA-21-Meis2-Nwd2.json', 'gaba_expr_mlog_df_c_GABA-3-Sst-Npy-Chodl.json', 'gaba_expr_mlog_df_c_GABA-15-Igsf1-Zfhx3.json', 'gaba_expr_mlog_df_c_GABA-27-Greb1-Dlk1.json', 'gaba_expr_mlog_df_c_GABA-6-Hapln1-Cryab.json', 'gaba_expr_mlog_df_c_GABA-19-Meis2-Tshz1.json', 'gaba_expr_mlog_df_c_GABA-25-Oprk1-Trhde.json', 'gaba_expr_mlog_df_c_GABA-28-Lypd1-Satb1.json', 'gaba_expr_mlog_df_c_GABA-11-Pax6-Calca.json', 'gaba_expr_mlog_df_c_GABA-32-Cartpt-Dkk2.json', 'gaba_expr_mlog_df_c_GABA-10-Pax6-Npnt.json', 'gaba_expr_mlog_df_c_GABA-4-Moxd1-Pvalb.js

In [13]:
files[4]

'gaba_expr_mlog_df_c_GABA-24-Prlr-St18.json'

In [4]:
test_df = pd.read_json(files[4])

In [25]:
test_df

,N_f,B_f,N_m,B_m
0610007P14Rik,0.627974,0.356680,0.562874,0.322593
0610009B22Rik,0.372232,0.221154,0.382525,0.224377
0610009L18Rik,0.141602,0.057692,0.111421,0.116675
0610009O20Rik,0.131893,0.126634,0.142105,0.070270
0610010F05Rik,0.269465,0.130625,0.345683,0.185621
...,...,...,...,...
Zyg11b,0.466607,0.601244,0.655332,0.445089
Zzef1,0.205533,0.320575,0.434365,0.270782
Zzz3,0.338172,0.410382,0.377176,0.335647
l7Rn6,0.529221,0.353412,0.551453,0.349620


In [5]:
delta_B_N_m = test_df['B_m'] - test_df['N_m']
delta_B_N_f = test_df['B_f'] - test_df['N_f']
delta_m_f_N = test_df['N_m'] - test_df['N_f']
delta_m_f_B = test_df['B_m'] - test_df['B_f']
delta_df = pd.DataFrame(index=test_df.index, columns=['delta_B_N_m',
                                                      'delta_B_N_f',
                                                      'delta_m_f_N',
                                                      'delta_m_f_B'])
delta_df.loc[:,'delta_B_N_m'] = delta_B_N_m
delta_df.loc[:,'delta_B_N_f'] = delta_B_N_f
delta_df.loc[:,'delta_m_f_B'] = delta_m_f_B
delta_df.loc[:,'delta_m_f_N'] = delta_m_f_N
delta_df

,delta_B_N_m,delta_B_N_f,delta_m_f_N,delta_m_f_B
0610007P14Rik,-0.240281,-0.271294,-0.0651,-0.034087
0610009B22Rik,-0.158148,-0.151078,0.010293,0.003223
0610009L18Rik,0.005255,-0.083909,-0.030181,0.058983
0610009O20Rik,-0.071835,-0.005259,0.010212,-0.056364
0610010F05Rik,-0.160062,-0.13884,0.076218,0.054996
...,...,...,...,...
Zyg11b,-0.210243,0.134637,0.188725,-0.156154
Zzef1,-0.163583,0.115042,0.228832,-0.049793
Zzz3,-0.041529,0.07221,0.039004,-0.074735
l7Rn6,-0.201833,-0.175809,0.022232,-0.003792


In [9]:
# Define the new directory name
density_dir = 'density_plots'
# Create the new directory
density_dir_path = os.path.join('/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run, density_dir)
os.makedirs(density_dir_path, exist_ok=True)

In [24]:
fig,ax = plt.subplots()
delta_df.plot(kind='scatter', x='delta_m_f_N', y='delta_m_f_B', ax=ax)
#for i, txt in enumerate(delta_df.index):
#    ax.annotate(txt, (delta_df.delta_m_f_B[i], delta_df.delta_m_f_N[i]))
opt_lim = np.abs(delta_df.values).max(axis=None) + 0.5
ax.axvline(color = 'grey')
ax.axhline(y=0, color = 'grey')
ax.set_xlim([-opt_lim,opt_lim])
ax.set_ylim([-opt_lim,opt_lim])
ax.set_box_aspect(1)
plt.savefig(density_dir_path + '/test.png')
plt.show()

<IPython.core.display.Javascript object>

In [17]:
opt_lim

1.4312265664

In [29]:
files[4].split('_')[-1].split('.')[0]

'GABA-24-Prlr-St18'

In [43]:
fig, axes = plt.subplots(1,2, figsize=(10, 5))
c = sns.histplot(delta_df, x="delta_B_N_m", y="delta_B_N_f", ax=axes[0])
d = sns.histplot(delta_df, x="delta_m_f_N", y="delta_m_f_B", ax=axes[1])

opt_lim = np.abs(delta_df.values).max(axis=None) + 0.5
axes[0].axvline(color = 'grey')
axes[0].axhline(y=0, color = 'grey')
c.set(xlim=(-opt_lim, opt_lim), ylim=(-opt_lim, opt_lim))

axes[1].axvline(color = 'grey')
axes[1].axhline(y=0, color = 'grey')
d.set(xlim=(-opt_lim, opt_lim), ylim=(-opt_lim, opt_lim))

plt.savefig(density_dir_path + '/displot_test.png')

plt.show()


<IPython.core.display.Javascript object>

In [ ]:
fig, ax = plt.subplots()
d = sns.kdeplot(data=delta_df, x="delta_m_f_N", y="delta_m_f_B", fill=True, ax=ax)
opt_lim = np.abs(delta_df.values).max(axis=None) + 0.5
#ax.axvline(color = 'grey')
#ax.axhline(y=0, color = 'grey')
ax.set(xlim=(-opt_lim, opt_lim), ylim=(-opt_lim, opt_lim))
ax.axvline(x=0, color='grey')
ax.axhline(y=0, color='grey')

plt.savefig(density_dir_path + '/kdeplot_test.png')
plt.show()


<IPython.core.display.Javascript object>

In [5]:
%%capture output
#cluster_fn = 'Vglut2-6-Otp-Zic5'
run = '040325_run'
cell_class = 'Nonneuronal'
delta_data_folder = '/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run+'/gene_delta_plots/data/'

# Define the new directory name
density_dir = 'density_plots'
# Create the new directory
density_dir_path = os.path.join('/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run, density_dir)
os.makedirs(density_dir_path, exist_ok=True)

# Change the current working directory to the specified path
os.chdir(delta_data_folder)

# Get a list of all files in the directory that include '_expr_mlog_df_c_'
files = [f for f in os.listdir() if '_expr_mlog_df_c_' in f]

# Print the list of files
print(files)

for f in files:
    test_df = pd.read_json(f)
    fn = f.split('_')[-1].split('.')[0]
    
    delta_B_N_m = test_df['B_m'] - test_df['N_m']
    delta_B_N_f = test_df['B_f'] - test_df['N_f']
    delta_m_f_N = test_df['N_m'] - test_df['N_f']
    delta_m_f_B = test_df['B_m'] - test_df['B_f']
    delta_df = pd.DataFrame(index=test_df.index, columns=['delta_B_N_m',
                                                        'delta_B_N_f',
                                                        'delta_m_f_N',
                                                        'delta_m_f_B'])
    delta_df.loc[:,'delta_B_N_m'] = delta_B_N_m
    delta_df.loc[:,'delta_B_N_f'] = delta_B_N_f
    delta_df.loc[:,'delta_m_f_B'] = delta_m_f_B
    delta_df.loc[:,'delta_m_f_N'] = delta_m_f_N

    fig, axes = plt.subplots(1,2, figsize=(10, 5))
    c = sns.histplot(delta_df, x="delta_B_N_m", y="delta_B_N_f", ax=axes[0])
    d = sns.histplot(delta_df, x="delta_m_f_N", y="delta_m_f_B", ax=axes[1])

    opt_lim = np.abs(delta_df.values).max(axis=None) + 0.5
    axes[0].axvline(color = 'grey')
    axes[0].axhline(y=0, color = 'grey')
    c.set(xlim=(-opt_lim, opt_lim), ylim=(-opt_lim, opt_lim))

    axes[1].axvline(color = 'grey')
    axes[1].axhline(y=0, color = 'grey')
    d.set(xlim=(-opt_lim, opt_lim), ylim=(-opt_lim, opt_lim))


    plt.savefig(density_dir_path + '/'+ fn +'_dualhistplot' + '.pdf') 

    plt.show()
    